In [72]:
!pip -q install transformers datasets peft trl accelerate bitsandbytes
!pip -q install sentence-transformers faiss-cpu
!pip -q install wandb

In [73]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [74]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets huggingface_hub


In [75]:
%cd /content

!git clone https://github.com/lm-playpen/playpen.git

%cd playpen

/content
fatal: destination path 'playpen' already exists and is not an empty directory.
/content/playpen


In [76]:
import os

for root, dirs, files in os.walk("."):
    level = root.count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:8]:
        print(f"{indent}    {f}")

./
    model_registry.json
    game_registry.json.template
    key.json.template
    SETUP.md
    README.md
    .gitignore
    LICENSE
    pyproject.toml
    tests/
        test_evaluate.py
    playpen/
        __init__.py
        cli.py
        buffers.py
        base.py
        callbacks/
            __init__.py
            buffers.py
        agents/
            __init__.py
            openenv.py
            clem.py
            base.py
        starters/
            sequential_trainer.py
            __init__.py
            batch_trainer.py
            branching_trainer.py
    examples/
        pettingzoo/
            taboo.ipynb
            wordle.ipynb
        trl/
            sft_trainer_lora.py
            sft_trainer_simple.py
            data_utils.py
            dpo_trainer.py
            playpen-qwen-lora/
        openenv/
            wordle-trl.ipynb
            taboo.ipynb
            wordle.ipynb
        gymnasium/
            taboo.ipynb
            wordle.ipynb
    .git/
 

In [77]:
%cd /content/playpen/examples/trl

!ls -lah

/content/playpen/examples/trl
total 32K
drwxr-xr-x 3 root root 4.0K Jul 12 15:29 .
drwxr-xr-x 6 root root 4.0K Jul 12 15:17 ..
-rw-r--r-- 1 root root 4.6K Jul 12 15:17 data_utils.py
-rw-r--r-- 1 root root 3.4K Jul 12 15:17 dpo_trainer.py
drwxr-xr-x 2 root root 4.0K Jul 12 15:29 playpen-qwen-lora
-rw-r--r-- 1 root root 2.7K Jul 12 15:17 sft_trainer_lora.py
-rw-r--r-- 1 root root 2.3K Jul 12 15:17 sft_trainer_simple.py


In [78]:
for f in [
    "data_utils.py",
    "sft_trainer_simple.py",
    "sft_trainer_lora.py",
    "dpo_trainer.py",
]:
    print("=" * 80)
    print(f)
    print("=" * 80)
    with open(f, "r") as file:
        lines = file.readlines()
    print("".join(lines[:120]))  # first 120 lines

data_utils.py
import argparse
import json
import os.path
import random
from glob import glob
from tqdm import tqdm

from clemcore.clemgame.resources import load_json


def create_conversational_dataset_for(top_dir):
    """NOTE: This script requires interactions generated with clemcore >=2.4.0 !"""
    interactions_files = glob(f"{top_dir}/**/interactions.json", recursive=True)
    dataset_file = "results.jsonl"
    dataset_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), dataset_file)
    print(f"Writing dataset file to {dataset_path} interactions")
    exceptions = set()
    with open(dataset_path, "w", encoding="utf-8") as f:
        print(f"Collecting {len(interactions_files)} interactions")
        for interactions_file in tqdm(interactions_files):
            interactions = load_json(interactions_file)
            # read from meta info (since clemcore 2.4)
            game_name = interactions["meta"]["game_name"]
            experiment_name = interactions["meta"]["

In [79]:
from datasets import load_dataset

dataset = load_dataset(
    "colab-potsdam/playpen-data",
    "interactions",
    split="train"
)

print(dataset)
print(dataset[0].keys())
print(dataset[0]["meta"])
print()

for message in dataset[0]["messages"]:
    print(message)

Dataset({
    features: ['messages', 'meta'],
    num_rows: 34909
})
dict_keys(['messages', 'meta'])
{'experiment': 'medium_en', 'game': 'taboo', 'game_role': 'WordDescriber', 'model': 'claude-sonnet-4-5-20250929', 'outcome': 'success', 'player_name': 'Player 1', 'task_id': 5}

{'content': 'You are playing a collaborative word guessing game in which you have to describe a target word for another player to guess.\n\nRules:\n(a) You have to reply in the form: CLUE: <some text>. Guesses from the other player will start with GUESS.\n(b) You cannot use the target word itself, parts or morphological variants of it in your description.\n(c) In addition, the same rules apply for related words which are provided below.\n\nEnd conditions:\n(i) If you use the target word or a related word in your description, then you lose.\n(ii) If the other player can guess the target word in 3 tries, you both win.\n\nLet us start.\n\nThis is the target word that you need to describe and that the other player n

In [80]:
from collections import Counter

outcomes = Counter()
games = Counter()

for sample in dataset:
    outcomes[sample["meta"]["outcome"]] += 1
    games[sample["meta"]["game"]] += 1

print("Outcomes")
print(outcomes)

print("\nGames")
for game, n in games.items():
    print(f"{game:20s} {n}")

Outcomes
Counter({'success': 20202, 'failure': 7794, 'aborted': 6913})

Games
taboo                3196
hot_air_balloon      1576
referencegame        5207
adventuregame        1085
wordle_withcritic    1480
wordle               837
privateshared        1395
dond                 2218
guesswhat            3180
textmapworld_graphreasoning 774
textmapworld         1395
matchit_ascii        2203
wordle_withclue      837
textmapworld_specificroom 837
imagegame            1897
codenames            6792


In [81]:
import numpy as np

lengths = [len(x["messages"]) for x in dataset]

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("Max:", np.max(lengths))
print("Min:", np.min(lengths))

Mean: 15.763843135008164
Median: 6.0
Max: 734
Min: 2


In [82]:
dataset = dataset.filter(
    lambda episode: episode["meta"]["outcome"] == "success"
)

In [83]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

MODEL = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [84]:
print(tokenizer.chat_template[:500])

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool 


In [85]:
max_length = 300

In [86]:
lengths = []

for sample in dataset.select(range(1000)):
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize=False
    )

    ids = tokenizer(text).input_ids
    lengths.append(len(ids))

import numpy as np

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("95th percentile:", np.percentile(lengths,95))
print("99th percentile:", np.percentile(lengths,99))
print("Max:", max(lengths))

Mean: 817.294
Median: 498.5
95th percentile: 2227.2999999999997
99th percentile: 5821.059999999992
Max: 10040


In [87]:
from datasets import load_dataset

dataset = load_dataset(
    "colab-potsdam/playpen-data",
    "interactions",
    split="train"
)

# Success only (baseline)
dataset = dataset.filter(
    lambda x: x["meta"]["outcome"] == "success"
)

In [88]:
MAX_TOKENS = 512

def keep_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False
    )

    n_tokens = len(tokenizer(text).input_ids)

    example["length"] = n_tokens

    return n_tokens <= MAX_TOKENS

dataset = dataset.filter(keep_example)

print(dataset)

Filter:   0%|          | 0/20202 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'meta'],
    num_rows: 11027
})


In [89]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [100]:
training_args = SFTConfig(
    output_dir="playpen-qwen-lora",

    max_length=512,          # ↓ from 2048

    num_train_epochs=1,

    learning_rate=2e-4,

    per_device_train_batch_size=2,   # ↑

    gradient_accumulation_steps=4,   # ↓ from 16

    logging_steps=20,

    save_strategy="epoch",           # don't save every 500 steps

    eval_strategy="epoch",           # evaluate once

    packing=False,

    completion_only_loss=True,

    bf16=torch.cuda.is_bf16_supported(),

    fp16=not torch.cuda.is_bf16_supported(),

    gradient_checkpointing=True,

    report_to="none",

    dataloader_num_workers=4

)

In [91]:
print("Train examples:", len(train_dataset))
print("Batch size:", training_args.per_device_train_batch_size)
print("Grad accumulation:", training_args.gradient_accumulation_steps)

effective_batch = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

steps_per_epoch = len(train_dataset) // effective_batch

print("Effective batch size:", effective_batch)
print("Optimizer steps per epoch:", steps_per_epoch)

Train examples: 8331
Batch size: 2
Grad accumulation: 4
Effective batch size: 8
Optimizer steps per epoch: 1041


In [101]:
from collections import defaultdict
import random
from datasets import Dataset

random.seed(42)

MAX_PER_GAME = 700

game_examples = defaultdict(list)

for ex in dataset:
    game_examples[ex["meta"]["game"]].append(ex)

balanced = []

for game, examples in game_examples.items():
    random.shuffle(examples)
    balanced.extend(examples[:MAX_PER_GAME])

random.shuffle(balanced)

dataset = Dataset.from_list(balanced)

print(dataset)

print("\nExamples per game")

from collections import Counter

counter = Counter()

for ex in dataset:
    counter[ex["meta"]["game"]] += 1

for game, n in sorted(counter.items()):
    print(f"{game:30s} {n}")

TypeError: string indices must be integers, not 'str'

In [93]:
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42,
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['messages', 'meta'],
    num_rows: 4207
})
Dataset({
    features: ['messages', 'meta'],
    num_rows: 468
})


In [94]:
print(training_args.max_length)

512


In [102]:
import numpy as np

lengths = []

for ex in train_dataset.select(range(min(1000, len(train_dataset)))):
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
    )
    lengths.append(len(tokenizer(text).input_ids))

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("95th:", np.percentile(lengths, 95))

Mean: 345.646
Median: 346.5
95th: 499.04999999999995


In [103]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/4207 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4207 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4207 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/468 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/468 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/468 [00:00<?, ? examples/s]

In [104]:
trainer.model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch,Training Loss,Validation Loss


In [99]:
!nvidia-smi

Sun Jul 12 16:01:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             34W /   70W |    5975MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
trainer.save_model("playpen-qwen-lora")
tokenizer.save_pretrained("playpen-qwen-lora")

In [ ]:
metrics = trainer.evaluate()
print(metrics)

import json

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)

messages = [
    {
        "role":"system",
        "content":"You are a helpful game-playing assistant."
    },
    {
        "role":"user",
        "content":"We are playing Taboo. Describe an elephant without using the word elephant."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(pipe(prompt)[0]["generated_text"])